# Ljung–Box Test in Python

A reusable implementation using `statsmodels` to test for autocorrelation in a time series (or model residuals) at one or several lags, with automatic decision-making.

**Key idea:** the Ljung–Box test asks whether autocorrelations across a set of lags are *jointly* zero.

- **H0:** ρ₁ = ρ₂ = ⋯ = ρₕ = 0 (no significant autocorrelation up to lag h)
- **H1:** at least one autocorrelation is non-zero

It's especially useful for checking **model residuals** — for residual diagnostics, a *high* p-value is generally what you want, since it means the residuals look like white noise.

## 1. The `ljung_box_test` Helper Function

This function runs the Ljung–Box test on a series (or a set of lags) and prints a decision for each lag tested.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.diagnostic import acorr_ljungbox


def ljung_box_test(
    series,
    lags=10,
    significance=0.05,
    name="Time Series"
):
    """
    Perform Ljung-Box test and make a statistical decision.

    Parameters
    ----------
    series : array-like
        Time-series data or model residuals.
    lags : int or list
        Number of lags to test.
    significance : float
        Significance level, e.g. 0.05.
    name : str
        Name of the series.
    """

    # Convert to pandas Series and remove missing values
    series = pd.Series(series).dropna()

    # Perform Ljung-Box test
    result = acorr_ljungbox(
        series,
        lags=lags,
        return_df=True
    )

    print("=" * 70)
    print(f"LJUNG-BOX TEST: {name}")
    print("=" * 70)

    print("\nResults:")
    print(result)

    print(f"\nSignificance Level (\u03b1): {significance}")

    print("\nHypotheses:")
    print("H0: No significant autocorrelation up to the tested lag(s).")
    print("H1: Significant autocorrelation exists at one or more tested lags.")

    print("\nDecision:")

    # Decision for each lag
    for lag, row in result.iterrows():

        p_value = row["lb_pvalue"]

        if p_value <= significance:
            decision = "Reject H0"
            conclusion = "Significant autocorrelation exists."
        else:
            decision = "Fail to Reject H0"
            conclusion = "No significant autocorrelation detected."

        print(
            f"Lag {lag}: "
            f"p-value = {p_value:.4f} -> "
            f"{decision} -> {conclusion}"
        )

    print("=" * 70)

    return result

## 2. Test a Simulated Time Series (White Noise)

Generate pure random noise. Since white noise has no serial structure, we expect large p-values and a **fail to reject H0** decision at every lag.

In [ ]:
np.random.seed(42)

data = np.random.normal(
    loc=0,
    scale=1,
    size=500
)

result = ljung_box_test(
    data,
    lags=10,
    significance=0.05,
    name="Random Noise"
)

For white-noise-like data, you will generally see large p-values. The decision will typically look like:

```
Lag 1: p-value = 0.XXXX -> Fail to Reject H0
Lag 2: p-value = 0.XXXX -> Fail to Reject H0
...
Lag 10: p-value = 0.XXXX -> Fail to Reject H0
```

**Interpretation:** there is insufficient evidence of significant autocorrelation at the tested lags.

## 3. Create a Time Series With Autocorrelation

To see the opposite behavior, let's simulate an AR(1) process:

$$X_t = 0.8\,X_{t-1} + \varepsilon_t$$

In [ ]:
from statsmodels.tsa.arima_process import ArmaProcess

np.random.seed(42)

# AR(1): X_t = 0.8 X_(t-1) + error
ar = np.array([1, -0.8])
ma = np.array([1])

ar_process = ArmaProcess(ar, ma)

autocorrelated_data = ar_process.generate_sample(
    nsample=500
)

result = ljung_box_test(
    autocorrelated_data,
    lags=10,
    significance=0.05,
    name="Autocorrelated Series"
)

You will typically get small p-values at several lags:

```
Lag 1: p-value = 0.0000 -> Reject H0
Lag 2: p-value = 0.0000 -> Reject H0
...
```

**Conclusion:** there is statistically significant autocorrelation in the series.

## 4. Testing Stock Returns

Load a CSV containing a `Close` price column, compute returns, and test for autocorrelation.

In [ ]:
df = pd.read_csv("stock_data.csv")

df["Return"] = df["Close"].pct_change()

result = ljung_box_test(
    df["Return"],
    lags=10,
    significance=0.05,
    name="Stock Returns"
)

You can also test several lags explicitly, which tests the joint autocorrelation up to each specified lag:

In [ ]:
result = ljung_box_test(
    df["Return"],
    lags=[5, 10, 20],
    significance=0.05,
    name="Stock Returns"
)

## 5. Most Important Application: Testing Model Residuals

Ljung–Box is especially useful after fitting a time-series model:

```
Time Series
     |
   ARIMA
     |
   Model
     |
  Residuals
     |
Ljung-Box Test
```

Fit an ARIMA model and test its residuals for remaining autocorrelation.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Fit an example ARIMA model on the autocorrelated series from Section 3
model = ARIMA(autocorrelated_data, order=(1, 0, 0)).fit()
residuals = model.resid

result = ljung_box_test(
    residuals,
    lags=[10, 20],
    significance=0.05,
    name="ARIMA Residuals"
)

If you get something like:

```
Lag 10: p-value = 0.42
Lag 20: p-value = 0.31
```

then:

```
p-value > 0.05
        |
Fail to Reject H0
        |
No significant autocorrelation detected
        |
Residuals are consistent with white noise
        |
Good sign for model adequacy
```

## 6. If the p-value Is Small

Suppose your model produces:

```
Lag 10: p-value = 0.002
```

Since 0.002 < 0.05, we **reject H0**.

**Conclusion:** significant autocorrelation remains in the residuals. This suggests that your model may not have captured all of the temporal structure. You may need to reconsider:

- AR terms
- MA terms
- Seasonal terms
- Differencing
- Model order
- Other time-series structure

## 7. Decision Rule

The Ljung–Box test has:

$$H_0: \rho_1 = \rho_2 = \cdots = \rho_h = 0$$
$$H_1: \text{at least one autocorrelation is non-zero}$$

At α = 0.05, the decision rule is:

| p-value | Decision | Interpretation |
|---|---|---|
| p ≤ 0.05 | Reject H0 | Significant autocorrelation exists |
| p > 0.05 | Fail to reject H0 | No significant autocorrelation detected |

### Important for residual diagnostics

For model residuals, you generally **want**:

```
p-value > 0.05
        |
Fail to Reject H0
        |
No significant residual autocorrelation
        |
Residuals approximately behave like white noise
```

So unlike some tests, a **high** Ljung–Box p-value is generally a desirable result when you're diagnosing model residuals.